# 01 – Data Acquisition: Hourly Temperature Observations (2012)

## Objective

This notebook documents the acquisition of raw hourly temperature observations
for Oklahoma (starting with Oklahoma City) for the year 2012.

Raw observational data is collected from multiple sources including:

- NOAA surface observation stations
- Oklahoma Mesonet stations
- Potential additional observational networks

The **data retrieval itself is performed by reusable Python scripts**
located in the `scripts/` directory. This notebook demonstrates how
those scripts are executed and verifies the results.

## Goals of this Stage

The purpose of this stage is **not analysis**.

Instead we focus on building a **reproducible data foundation** for the
Weather Agreement Lab.

Specifically we will:

- Retrieve reproducible raw data
- Preserve the original files exactly as received
- Store the files in `data/raw/`
- Document station metadata and acquisition details

## Role in the Weather Agreement Lab Pipeline

This notebook establishes the **data acquisition layer** of the project.

Downstream stages will:

- Clean and align the observational datasets
- Measure agreement between agencies
- Explore forecast uncertainty
- Develop machine-learning based fusion models


# Suggested Updated Version

```
# Environment Setup (Run Before Executing This Notebook)

This notebook is part of the **Weather Agreement Lab**, a reproducible
data science workflow for studying agreement between weather observation
networks.

The project follows a **research-style pipeline**:

```

```
scripts   → retrieve and process data
notebooks → explore and explain results
report    → produce the final LaTeX write-up
```

```

Before running this notebook, create a Python virtual environment and
install the required dependencies.

---

## 1. Clone the Repository

```

```bash
git clone git@github.com:jeremy-evert/handson-ml3.git
cd handson-ml3/Weather_Agreement_Lab
```

---

## 2. Create a Virtual Environment

Linux / macOS

```bash
python3 -m venv .venv
```

Windows

```powershell
python -m venv .venv
```

---

## 3. Activate the Virtual Environment

Linux / macOS

```bash
source .venv/bin/activate
```

Windows

```powershell
.venv\Scripts\activate
```

---

## 4. Install Required Packages

````

```bash
pip install --upgrade pip
pip install -r requirements_weather_lab.txt
````

---

## 5. Register the Jupyter Kernel

This allows the notebook to run using the virtual environment.

```bash
python -m ipykernel install --user \
  --name weather_lab_env \
  --display-name "Python (Weather Lab)"
```

---

## 6. Open the Project

Recommended workflow using **VS Code**:

```bash
code .
```

Then:

1. Open `01_data_pull.ipynb`
2. Click **Select Kernel** (upper right)
3. Choose **Python (Weather Lab)**

---

## Alternative: Launch Jupyter Directly

```bash
jupyter notebook
```

Open:

```
01_data_pull.ipynb
```

---

## Key Dependencies

Defined in:

```
requirements_weather_lab.txt
```

Major packages used in this lab include:

* pandas
* numpy
* matplotlib
* scikit-learn
* requests
* jupyter / ipykernel

---

## Project Layout

```
Weather_Agreement_Lab
│
├── notebooks
│   ├── 01_data_pull.ipynb
│   ├── 02_alignment_and_cleaning.ipynb
│   ├── 03_forecast_uncertainty.ipynb
│   ├── 04_agreement_metrics.ipynb
│   └── 05_data_fusion.ipynb
│
├── scripts
│   ├── pull_noaa.py
│   ├── clean_data.py
│   ├── compute_metrics.py
│   └── fusion_model.py
│
├── data
│   ├── raw
│   ├── processed
│   └── analysis
│
└── requirements_weather_lab.txt
```

Notebook progression:

1️⃣ Data Acquisition
2️⃣ Cleaning & Alignment
3️⃣ Forecast Uncertainty
4️⃣ Agreement Metrics
5️⃣ Data Fusion

In [85]:
import sys
import pandas as pd

print("Python executable:")
print(sys.executable)

Python executable:
/mnt/nora/git/handson-ml3/Weather_Agreement_Lab/.venv/bin/python


In [88]:
if ".venv" not in sys.executable:
    print("⚠️ WARNING: You are not running inside the project virtual environment.")

In [89]:
from scripts.session_diagnostics import run_session_diagnostics

run_session_diagnostics()

===== SESSION DIAGNOSTICS =====

Timestamp: 2026-03-11 14:45:30.862592

--- Python ---
Python version: 3.11.13 (main, Jan 16 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-11)]
Python executable: /mnt/nora/git/handson-ml3/Weather_Agreement_Lab/.venv/bin/python

--- Virtual Environment ---
VIRTUAL_ENV: /mnt/nora/git/handson-ml3/Weather_Agreement_Lab/.venv

--- Working Directory ---
Current working directory: /mnt/nora/git/handson-ml3/Weather_Agreement_Lab

--- Platform ---
System: Linux
Release: 5.14.0-611.34.1.el9_7.x86_64
Machine: x86_64
Processor: x86_64

--- Installed Key Packages ---
pandas: 3.0.1
requests: 2.32.5
python-dotenv: 1.2.2

--- Git Status ---
Git branch: main



In [1]:
from scripts.config import DATA_RAW, DATA_PROCESSED
from scripts.config import ensure_directories
ensure_directories()

print("Raw data directory:")
print(DATA_RAW)

Raw data directory:
/mnt/nora/git/handson-ml3/Weather_Agreement_Lab/data/raw


In [86]:
from scripts.pull_noaa import download_noaa_hourly

ImportError: cannot import name 'download_noaa_hourly' from 'scripts.pull_noaa' (/mnt/nora/git/handson-ml3/Weather_Agreement_Lab/scripts/pull_noaa.py)

In [87]:
download_noaa_hourly(
    station="OKC",
    year=2012,
    output_dir="data/raw"
)


NameError: name 'download_noaa_hourly' is not defined

In [58]:
# Core imports for the Weather Agreement Lab

import sys
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [59]:
import sys
print(sys.executable)

/mnt/nora/git/handson-ml3/Weather_Agreement_Lab/.venv/bin/python


In [60]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

assert ".venv" in sys.executable

Python: 3.11.13 (main, Jan 16 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-11)]
Executable: /mnt/nora/git/handson-ml3/Weather_Agreement_Lab/.venv/bin/python


# NOAA API Token Setup

This notebook retrieves weather observations from the NOAA Climate Data API.

To access the API you must obtain a **free NOAA API token**.

## Step 1 — Request a Token

Visit:

https://www.ncdc.noaa.gov/cdo-web/token

Fill out the short form and NOAA will email you an API token.

---

## Step 2 — Create a `.env` File

Inside the **Weather_Agreement_Lab** directory, create a file named:

```

.env

```

Example directory structure:

```

Weather_Agreement_Lab
│
├── .env
├── 01_data_pull.ipynb
├── 02_alignment_and_cleaning.ipynb
├── requirements_weather_lab.txt

```

---

## Step 3 — Add Your Token

Inside the `.env` file add:

```

NOAA_API_TOKEN=your_token_goes_here

```

Example:

```

NOAA_API_TOKEN=ABCD1234EFGH5678

```

---

## Step 4 — Keep the Token Private

The `.env` file should **never be committed to Git**.

Make sure `.gitignore` contains:

```

.env

````

---

## Step 5 — Run the Notebook

The notebook will automatically load the token using:

```python
from dotenv import load_dotenv
load_dotenv()
````

If the token is missing, the notebook will raise an error to prevent API requests from failing silently.

````

---

# 2️⃣ Fix the `dotenv` error

Your environment simply needs the package.

Inside your activated venv run:

```bash
pip install python-dotenv
````

Then your code will work:

```python
from dotenv import load_dotenv
import os

load_dotenv()

NOAA_API_TOKEN = os.getenv("NOAA_API_TOKEN")

if NOAA_API_TOKEN is None:
    raise ValueError("NOAA_API_TOKEN not found. Did you create a .env file?")

headers = {
    "token": NOAA_API_TOKEN
}
```

---

# 3️⃣ Add this to your requirements file

Since the notebook uses it, make sure the requirements include it:

```
python-dotenv>=1.0
```

---

# 4️⃣ Small improvement (very helpful for students)

I recommend printing confirmation when it works:

```python
print("NOAA token loaded successfully.")
```

Students immediately know the environment is configured correctly.

---

💡 **Optional improvement I strongly recommend**

For teaching, I usually also include this cell:

```python
print("Token loaded:", NOAA_API_TOKEN[:4] + "...")
```

It confirms the token loaded **without exposing the full key**.

---


In [61]:
from dotenv import load_dotenv
import os

load_dotenv()

NOAA_API_TOKEN = os.getenv("NOAA_API_TOKEN")
print("NOAA token loaded successfully.")
print("Token loaded:", NOAA_API_TOKEN[:4] + "...")

if NOAA_API_TOKEN is None:
    raise ValueError("NOAA_API_TOKEN not found. Did you create a .env file?")

headers = {
    "token": NOAA_API_TOKEN
}

NOAA token loaded successfully.
Token loaded: dquB...


In [62]:
import requests
test_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datasets"

response = requests.get(test_url, headers=headers)

print("Status Code:", response.status_code)

Status Code: 200


In [63]:
from pathlib import Path

root = Path.cwd()
raw = root / "data" / "raw"
processed = root / "data" / "processed"

print("Root:", root)
print("Raw:", raw.exists())
print("Processed:", processed.exists())
print("Raw contents:", list(raw.glob("*")))

Root: /mnt/nora/git/handson-ml3/Weather_Agreement_Lab
Raw: True
Processed: True
Raw contents: [PosixPath('/mnt/nora/git/handson-ml3/Weather_Agreement_Lab/data/raw/noaa_okc_2012_hourly.csv'), PosixPath('/mnt/nora/git/handson-ml3/Weather_Agreement_Lab/data/raw/noaa_okc_2012.gz')]


In [64]:
import os

os.makedirs("data/raw", exist_ok=True)

In [65]:
import os

print("Working directory:", os.getcwd())
print("\nDirectory contents:")
print(os.listdir())

Working directory: /mnt/nora/git/handson-ml3/Weather_Agreement_Lab

Directory contents:
['04_agreement_metrics.ipynb', '.venv', 'requirements.txt', 'data', '.env', '03_forecast_uncertainty.ipynb', '02_alignment_and_cleaning.ipynb', '.ipynb_checkpoints', '05_data_fusion.ipynb', 'requirements_weather_lab.txt', 'README.md', '01_data_pull.ipynb']


In [66]:
print(os.listdir("data"))

['processed', 'raw', 'analysis']


In [67]:
import pathlib

for p in pathlib.Path(".").rglob("*okc*"):
    print(p)

data/raw/noaa_okc_2012_hourly.csv
data/raw/noaa_okc_2012.gz


In [68]:
import os

folders = [
    "data/raw",
    "data/processed",
    "data/analysis"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Data directories created.")

Data directories created.


In [69]:
import pandas as pd

#stations_url = "https://www.ncei.noaa.gov/data/global-hourly/doc/isd-history.csv"
stations_url = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"

stations = pd.read_csv(stations_url)

print(stations.head())
print(stations.columns)

     USAF   WBAN STATION NAME CTRY STATE ICAO    LAT     LON  ELEV(M)  \
0  007018  99999   WXPOD 7018  NaN   NaN  NaN   0.00   0.000   7018.0   
1  007026  99999   WXPOD 7026   AF   NaN  NaN   0.00   0.000   7026.0   
2  007070  99999   WXPOD 7070   AF   NaN  NaN   0.00   0.000   7070.0   
3  008260  99999    WXPOD8270  NaN   NaN  NaN   0.00   0.000      0.0   
4  008268  99999    WXPOD8278   AF   NaN  NaN  32.95  65.567   1156.7   

      BEGIN       END  
0  20110309  20130730  
1  20120713  20170822  
2  20140923  20150926  
3  20050101  20120731  
4  20100519  20120323  
Index(['USAF', 'WBAN', 'STATION NAME', 'CTRY', 'STATE', 'ICAO', 'LAT', 'LON',
       'ELEV(M)', 'BEGIN', 'END'],
      dtype='str')


In [70]:
okc = stations[
    stations["STATION NAME"].str.contains("OKLAHOMA CITY", case=False, na=False)
]

print(okc[["USAF","WBAN","STATION NAME","BEGIN","END"]])

         USAF   WBAN                   STATION NAME     BEGIN       END
17582  720628  99999             OKLAHOMA CITY/PAGE  20090312  20090325
19125  723544  99999            OKLAHOMA CITY/WILEY  19801023  19971231
28635  999999  13919       OKLAHOMA CITY TINKER AAF  19710101  19710101
28651  999999  13967  OKLAHOMA CITY WILL ROGERS FIE  19650101  19721231


In [71]:
import pandas as pd

stations_url = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"

stations = pd.read_csv(stations_url)

print(stations.head())
print(stations.columns)

okc = stations[
    stations["STATION NAME"].str.contains("WILL ROGERS", case=False, na=False)
]

print(okc[["USAF","WBAN","STATION NAME","BEGIN","END"]])

     USAF   WBAN STATION NAME CTRY STATE ICAO    LAT     LON  ELEV(M)  \
0  007018  99999   WXPOD 7018  NaN   NaN  NaN   0.00   0.000   7018.0   
1  007026  99999   WXPOD 7026   AF   NaN  NaN   0.00   0.000   7026.0   
2  007070  99999   WXPOD 7070   AF   NaN  NaN   0.00   0.000   7070.0   
3  008260  99999    WXPOD8270  NaN   NaN  NaN   0.00   0.000      0.0   
4  008268  99999    WXPOD8278   AF   NaN  NaN  32.95  65.567   1156.7   

      BEGIN       END  
0  20110309  20130730  
1  20120713  20170822  
2  20140923  20150926  
3  20050101  20120731  
4  20100519  20120323  
Index(['USAF', 'WBAN', 'STATION NAME', 'CTRY', 'STATE', 'ICAO', 'LAT', 'LON',
       'ELEV(M)', 'BEGIN', 'END'],
      dtype='str')
         USAF   WBAN                         STATION NAME     BEGIN       END
15152  700260  27502  W POST-WILL ROGERS MEMORIAL AIRPORT  19450101  20250827
19115  723530  13967            WILL ROGERS WORLD AIRPORT  19411214  20250827
28651  999999  13967        OKLAHOMA CITY WILL ROGE

In [72]:
import requests
import gzip
import shutil
import os

year = "2012"
station = "723530-13967"

url = f"https://www.ncei.noaa.gov/pub/data/noaa/isd-lite/{year}/{station}-{year}.gz"

print("Downloading:", url)

os.makedirs("data/raw", exist_ok=True)

gz_path = f"data/raw/noaa_okc_{year}.gz"
csv_path = f"data/raw/noaa_okc_{year}_hourly.csv"

r = requests.get(url)

print("HTTP status:", r.status_code)

if r.status_code == 200:
    with open(gz_path, "wb") as f:
        f.write(r.content)

    with gzip.open(gz_path, "rb") as f_in:
        with open(csv_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

    print("Saved:", csv_path)

else:
    print("Download failed")

Downloading: https://www.ncei.noaa.gov/pub/data/noaa/isd-lite/2012/723530-13967-2012.gz
HTTP status: 200
Saved: data/raw/noaa_okc_2012_hourly.csv


In [73]:
import pandas as pd

df = pd.read_csv("data/raw/noaa_okc_2012_hourly.csv", low_memory=False)

print(df.shape)
print(df.columns)
print(df.head())

(8783, 1)
Index(['2012 01 01 00   167     6 10119   340   123     2 -9999 -9999'], dtype='str')
  2012 01 01 00   167     6 10119   340   123     2 -9999 -9999
0  2012 01 01 01   150   -39 10144   330   108   ...           
1  2012 01 01 02   133   -67 10161   320    93   ...           
2  2012 01 01 03   111   -22 10186   310   144   ...           
3  2012 01 01 04    83    -6 10204   330   165   ...           
4  2012 01 01 05    72    -6 10221   330   124   ...           


In [74]:
import pandas as pd
import os
print(os.getcwd())

filepath = "data/raw/noaa_okc_2012_hourly.csv"

import os

if not os.path.exists(filepath):
    print("File missing. Run the download cell first.")

df = pd.read_csv(filepath, low_memory=False)

print(df.head())
print(df.columns)
print(df.shape)

/mnt/nora/git/handson-ml3/Weather_Agreement_Lab
  2012 01 01 00   167     6 10119   340   123     2 -9999 -9999
0  2012 01 01 01   150   -39 10144   330   108   ...           
1  2012 01 01 02   133   -67 10161   320    93   ...           
2  2012 01 01 03   111   -22 10186   310   144   ...           
3  2012 01 01 04    83    -6 10204   330   165   ...           
4  2012 01 01 05    72    -6 10221   330   124   ...           
Index(['2012 01 01 00   167     6 10119   340   123     2 -9999 -9999'], dtype='str')
(8783, 1)


In [81]:
import pandas as pd

filepath = "data/raw/noaa_okc_2012_hourly.csv"

df = pd.read_csv(filepath, delim_whitespace=True, header=None)

print(df.head())
print(df.shape)

TypeError: read_csv() got an unexpected keyword argument 'delim_whitespace'

In [75]:
print(df.columns)

Index(['2012 01 01 00   167     6 10119   340   123     2 -9999 -9999'], dtype='str')


In [76]:
for col in df.columns:
    print(col)

2012 01 01 00   167     6 10119   340   123     2 -9999 -9999


In [77]:
df.head(5)

,2012 01 01 00 167 6 10119 340 123 2 -9999 -9999
0,2012 01 01 01 150 -39 10144 330 108 ...
1,2012 01 01 02 133 -67 10161 320 93 ...
2,2012 01 01 03 111 -22 10186 310 144 ...
3,2012 01 01 04 83 -6 10204 330 165 ...
4,2012 01 01 05 72 -6 10221 330 124 ...


In [78]:
print(df.columns)

Index(['2012 01 01 00   167     6 10119   340   123     2 -9999 -9999'], dtype='str')


In [79]:
"TEMP" in df.columns
"TMP" in df.columns

False

In [80]:
def parse_temp(tmp_string):
    try:
        value = tmp_string.split(",")[0]
        return int(value) / 10
    except:
        return None

df["temperature_C"] = df["TMP"].apply(parse_temp)

df[["DATE","temperature_C"]].head()

KeyError: 'TMP'

In [ ]:
import os

filepath = "data/raw/noaa_okc_2012_hourly.txt"

if not os.path.exists(filepath):
    raise FileNotFoundError(
        f"{filepath} not found. Did you run the data download step?"
    )

In [ ]:
import pandas as pd

df = pd.read_csv(
    "data/raw/noaa_okc_2012_hourly.txt",
    sep=",",
    low_memory=False
)

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:

import pandas as pd

df = pd.read_csv("data/raw/noaa_okc_2012_hourly.txt")

df.info()

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

token = os.getenv("NOAA_TOKEN")

url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

params = {
    "datasetid": "GHCND",
    "locationid": "FIPS:40",   # Oklahoma
    "limit": 10
}

headers = {
    "token": token
}

response = requests.get(url, headers=headers, params=params)

stations = response.json()

print("Status:", response.status_code)
print("Keys returned:", stations.keys())
print("Number of stations returned:", len(stations.get("results", [])))

for s in stations.get("results", []):
    print(s["id"], "-", s["name"])

In [ ]:
base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"



params = {
    "datasetid": "GHCND",
    "locationid": "FIPS:40",
    "limit": 1000
}

response = requests.get(base_url, headers=headers, params=params)

stations = response.json()
stations_df = pd.DataFrame(stations["results"])

stations_df.head()

In [ ]:
stations_df[stations_df["name"].str.contains("OKLAHOMA CITY", case=False)]

In [ ]:
import requests

base_dir = "https://www.ncei.noaa.gov/data/global-hourly/archive/csv/2012/"

response = requests.get(base_dir)
print("Status:", response.status_code)

print(response.text[:2000])

In [ ]:
if response.status_code == 200:
    if "723530" in response.text:
        print("Found 723530 in directory listing!")
    else:
        print("723530 not found in listing.")

In [ ]:
import os
import requests

url = "https://www.ncei.noaa.gov/pub/data/noaa/2012/723530-13967-2012.gz"
save_path = "data/raw/noaa_okc_2012_hourly.gz"

print("Downloading from:", url)

os.makedirs("data/raw", exist_ok=True)

response = requests.get(url)
print("Status code:", response.status_code)

response.raise_for_status()

with open(save_path, "wb") as f:
    f.write(response.content)

print("Download complete.")
print("File size (bytes):", os.path.getsize(save_path))

In [ ]:
import gzip

source_path = "data/raw/noaa_okc_2012_hourly.gz"
decompressed_path = "data/raw/noaa_okc_2012_hourly.txt"

with gzip.open(source_path, 'rb') as f_in:
    with open(decompressed_path, 'wb') as f_out:
        f_out.write(f_in.read())

print("Decompressed file created.")
print("File size:", os.path.getsize(decompressed_path), "bytes")

In [ ]:
with open(decompressed_path, 'r') as f:
    for _ in range(5):
        print(f.readline())

In [ ]:
# Count number of lines in the decompressed NOAA file

file_path = "data/raw/noaa_okc_2012_hourly.txt"

line_count = 0

with open(file_path, "r") as f:
    for _ in f:
        line_count += 1

print("Number of lines in file:", line_count)

In [ ]:
from datetime import datetime

start = datetime(2012, 1, 1, 0, 0, 0)
end   = datetime(2013, 1, 1, 0, 0, 0)

hours_2012 = int((end - start).total_seconds() / 3600)

print("Total hours in 2012:", hours_2012)

In [ ]:
print("Lines in NOAA file:", line_count)
print("Expected hours in 2012:", hours_2012)
print("Difference:", line_count - hours_2012)

In [ ]:
from datetime import datetime
import pandas as pd

file_path = "data/raw/noaa_okc_2012_hourly.txt"

timestamps = []

with open(file_path, "r") as f:
    for line in f:
        dt_str = line[15:27]  # YYYYMMDDHHMM
        try:
            dt = datetime.strptime(dt_str, "%Y%m%d%H%M")
            timestamps.append(dt)
        except:
            continue

print("Parsed timestamps:", len(timestamps))
timestamps[:5]

In [ ]:
unique_hours = pd.Series(timestamps).dt.floor("h").nunique()

print("Unique hourly timestamps:", unique_hours)
print("Expected hours:", 8784)

In [ ]:
temps = []
parsed_datetimes = []

with open(file_path, "r") as f:
    for line in f:
        dt_str = line[15:27]
        tmp_str = line[87:92]  # may adjust if needed
        
        try:
            dt = datetime.strptime(dt_str, "%Y%m%d%H%M")
            
            if tmp_str.startswith(("+", "-")):
                temp_c = int(tmp_str[0:5]) / 10.0
            else:
                continue
            
            parsed_datetimes.append(dt)
            temps.append(temp_c)
            
        except:
            continue

df = pd.DataFrame({
    "datetime": parsed_datetimes,
    "temp_C": temps
})

print("Parsed rows:", len(df))
df.head()

In [ ]:
df["hour"] = df["datetime"].dt.floor("h")

hourly_df = df.groupby("hour")["temp_C"].mean().reset_index()

print("Hourly rows:", len(hourly_df))
hourly_df.head()

In [ ]:
print("Expected:", 8784)
print("Actual unique hourly records:", len(hourly_df))

In [ ]:
hourly_stats = df.groupby("hour")["temp_C"].agg(["mean", "std", "count"]).reset_index()

hourly_stats.head()

In [ ]:
full_range = pd.date_range(
    start="2012-01-01 00:00:00",
    end="2012-12-31 23:00:00",
    freq="h"
)

missing = set(full_range) - set(hourly_stats["hour"])

print("Missing hours:", len(missing))

In [ ]:
hourly_stats["mean"].describe()

In [ ]:
q1 = hourly_stats["mean"].quantile(0.25)
q3 = hourly_stats["mean"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 3 * iqr
upper_bound = q3 + 3 * iqr

outliers = hourly_stats[
    (hourly_stats["mean"] < lower_bound) |
    (hourly_stats["mean"] > upper_bound)
]

print("Extreme outliers:", len(outliers))
outliers.head()

In [ ]:
hourly_stats["delta"] = hourly_stats["mean"].diff()

spikes = hourly_stats[hourly_stats["delta"].abs() > 15]

print("Large hour-to-hour jumps:", len(spikes))
spikes.head()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(hourly_stats["mean"], bins=50)
plt.title("Temperature Distribution (2012)")
plt.xlabel("Temp °C")
plt.ylabel("Frequency")
plt.show()

## Dataset 2: METAR Airport Weather Observations

### Objective

Our second weather dataset will come from **METAR aviation weather observations**.

METAR reports are standardized hourly weather observations produced by airports
around the world. These reports include information such as:

- temperature
- dew point
- wind speed and direction
- visibility
- precipitation
- atmospheric pressure
- cloud coverage

For this project we will retrieve METAR observations from:

**Station:** KOKC  
**Location:** Will Rogers World Airport (Oklahoma City)

### Why Are We Collecting This Data?

This dataset will allow us to compare multiple independent weather sources.

Our project goal is to study **agreement between weather data sources**, which means
we want to measure how often different systems report similar or conflicting weather
conditions.

We currently have:

| Source | Type |
|------|------|
NOAA ISD | historical weather archive |
METAR | airport observations |
Forecast API | predicted weather |

METAR will serve as an **independent observational dataset** that we can later compare
against the NOAA dataset and forecast models.

### Output

The dataset will be saved to:


data/raw/metar_okc_2012.csv


We are intentionally storing raw data without modification so that the original
source data is preserved.


### Plan for Retrieving METAR Data

To retrieve the METAR dataset we will use a publicly available data service
provided by the **Iowa Environmental Mesonet (IEM)**.

This service allows us to download historical METAR observations as CSV files.

The process will involve the following steps:

1. Define the weather station we want to analyze
2. Define the date range of interest (2012)
3. Construct a request to the Mesonet data service
4. Send the request using Python's `requests` library
5. Save the returned CSV data to our raw data directory
6. Verify that the dataset downloaded correctly

Each of these steps will be implemented carefully so that the notebook
remains readable and reproducible.

### Step 1: Define Request Parameters

Before we download the METAR dataset we must define the parameters of the request.

The important parameters include:

- **Station ID** — KOKC (Oklahoma City airport)
- **Date Range** — January 1, 2012 through December 31, 2012
- **Output Format** — CSV
- **Timezone** — UTC

We will also define the location where the dataset will be stored so that
our project maintains a consistent directory structure.

In [ ]:
from pathlib import Path

# ----------------------------------------
# Define project paths
# ----------------------------------------

raw_data_directory = Path("data/raw")

# ----------------------------------------
# Define METAR dataset parameters
# ----------------------------------------

metar_station_identifier = "KOKC"

metar_start_year = 2012
metar_start_month = 1
metar_start_day = 1

metar_end_year = 2012
metar_end_month = 12
metar_end_day = 31

# ----------------------------------------
# Define output file location
# ----------------------------------------

metar_output_file = raw_data_directory / "metar_okc_2012.csv"

print("Station:", metar_station_identifier)
print("Date range:", f"{metar_start_year}-01-01 through {metar_end_year}-12-31")
print("Output file:", metar_output_file)

### Explanation of the Parameter Setup

In the previous step we defined all parameters required for the METAR request.

Several design choices were intentional:

**Readable variable names**

Instead of short names like `s` or `yr`, we use descriptive variable names
such as `metar_station_identifier` and `metar_start_year`. This improves
readability for students and collaborators.

**Explicit date components**

Breaking the date into year/month/day components makes the request parameters
clear and easy to modify later if we wish to analyze additional years.

**Centralized output path**

All raw datasets are stored inside:


data/raw/


Maintaining a strict directory structure ensures that later notebooks
(`02_clean_and_align.ipynb`) can reliably locate input datasets.


### Step 1: Define Request Parameters

Before we download the METAR dataset we must define the parameters of the request.

The important parameters include:

- **Station ID** — KOKC (Oklahoma City airport)
- **Date Range** — January 1, 2012 through December 31, 2012
- **Output Format** — CSV
- **Timezone** — UTC

We will also define the location where the dataset will be stored so that
our project maintains a consistent directory structure.

In [ ]:
from pathlib import Path

# ----------------------------------------
# Define project paths
# ----------------------------------------

raw_data_directory = Path("data/raw")

# ----------------------------------------
# Define METAR dataset parameters
# ----------------------------------------

metar_station_identifier = "KOKC"

metar_start_year = 2012
metar_start_month = 1
metar_start_day = 1

metar_end_year = 2012
metar_end_month = 12
metar_end_day = 31

# ----------------------------------------
# Define output file location
# ----------------------------------------

metar_output_file = raw_data_directory / "metar_okc_2012.csv"

print("Station:", metar_station_identifier)
print("Date range:", f"{metar_start_year}-01-01 through {metar_end_year}-12-31")
print("Output file:", metar_output_file)

### Explanation of the Parameter Setup

In the previous step we defined all parameters required for the METAR request.

Several design choices were intentional:

**Readable variable names**

Instead of short names like `s` or `yr`, we use descriptive variable names
such as `metar_station_identifier` and `metar_start_year`. This improves
readability for students and collaborators.

**Explicit date components**

Breaking the date into year/month/day components makes the request parameters
clear and easy to modify later if we wish to analyze additional years.

**Centralized output path**

All raw datasets are stored inside:


data/raw/


Maintaining a strict directory structure ensures that later notebooks
(`02_clean_and_align.ipynb`) can reliably locate input datasets.


### Step 2: Construct the METAR Data Request

Now that the parameters are defined we will construct the request
that will be sent to the Mesonet METAR archive service.

This request will include:

- station identifier
- date range
- desired output format
- additional metadata such as latitude, longitude, and elevation

Once the request is constructed we will send it to the server
and retrieve the dataset.

In [ ]:
import requests

mesonet_request_url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

metar_request_parameters = {

    "station": metar_station_identifier,

    "data": "all",

    "year1": metar_start_year,
    "month1": metar_start_month,
    "day1": metar_start_day,

    "year2": metar_end_year,
    "month2": metar_end_month,
    "day2": metar_end_day,

    "tz": "UTC",
    "format": "onlycomma",

    "latlon": "yes",
    "elev": "yes",

    "missing": "M",
    "trace": "T",

    "direct": "yes",

    "report_type": "1",
    "report_type": "2"
}

print("METAR request parameters constructed successfully.")

### Understanding the METAR Request

The parameters provided to the Mesonet service control exactly what
data will be returned.

Important options include:

**station**

The airport weather station identifier.

Example:


KOKC


**format = onlycomma**

Requests a clean comma-separated file so the data can easily be
loaded into pandas.

**latlon = yes**

Includes geographic coordinates for the station.

**elev = yes**

Includes station elevation which may be useful for analysis.

The remaining parameters ensure that the service returns a
complete dataset without filtering out any weather observations.



In [ ]:
metar_response = requests.get(
    mesonet_request_url,
    params=metar_request_parameters
)

print("Server response code:", metar_response.status_code)

if metar_response.status_code == 200:

    with open(metar_output_file, "w", encoding="utf-8") as file_handle:
        file_handle.write(metar_response.text)

    print("METAR dataset successfully saved.")

else:
    print("Download failed.")

### Step 3: Verify the Downloaded Dataset

After retrieving the METAR dataset we should verify that the file
was downloaded correctly and contains usable data.

We will load the dataset into pandas and inspect:

- number of rows
- number of columns
- sample records

This step ensures that our data pipeline is functioning properly
before we move on to the next data source.

In [ ]:
import pandas as pd

metar_dataframe = pd.read_csv(metar_output_file)

print("Dataset shape:", metar_dataframe.shape)

metar_dataframe.head()